# MuCoCo RQ3 Experiment Results Aggregation for Mutation Operators

This notebook is used to aggregate the results for MuCoCo results shown in Figure 2. The results are stored in MuCoCo_results/MuCoCo_experiment_results/ in the project root folder. The final aggregated results from this notebook are used in Figure 2 in the report (consistency error rate and accuracy by mutation operator).

In [1]:
import os
import sys
import pandas as pd
from typing import Tuple, Dict
from pathlib import Path

In [2]:
curr_dir = Path(os.getcwd())
parent_dir = curr_dir.parent
proj_dir = parent_dir.parent
sys.path.append(str(proj_dir))

In [3]:
from utility.data_log_functions import DataLogHelper

/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def compare_multiple_code_generation_logs(res_dir: str, filter: Tuple[str] = (), anti_filter: Tuple[str] = ()):
    
    if filter is None:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv"))]
    else:
        csv_logs = [f for f in os.listdir(res_dir) if (
            os.path.isfile(os.path.join(res_dir, f)) and 
            f.endswith(".csv") and 
            all(sub in f for sub in filter)) and
            all(sub not in f for sub in anti_filter)
            ]

    log_file_names = [csv_file_name.replace('.csv', '') for csv_file_name in csv_logs]

    results_df = pd.DataFrame(columns=log_file_names, index = log_file_names)
    for file_name in log_file_names:
        results_df.loc[file_name, file_name] = float('nan')

    while len(csv_logs) > 0:
        log1_file_name = csv_logs.pop()
        for log2_file_name in csv_logs:
            log1_file_path = os.path.join(res_dir, log1_file_name)
            log2_file_path = os.path.join(res_dir, log2_file_name)

            log1 = pd.read_csv(log1_file_path)
            log2 = pd.read_csv(log2_file_path) 
            log1, log2 = DataLogHelper.standardize_two_df(log1, log2)
            log1_inconsistencies, log2_inconsistencies = DataLogHelper.compare_code_generation_dataframe_results(log1=log1, log2=log2)

            results_df.loc[log1_file_name.replace('.csv', ''), log2_file_name.replace('.csv', '')] = log1_inconsistencies
            results_df.loc[log2_file_name.replace('.csv', ''), log1_file_name.replace('.csv', '')] = log2_inconsistencies

    return results_df

In [5]:
def compare_logs_against_no_mutation(
        res_dir: str, 
        benchmark: str,
        task: str,
        filter: Tuple[str] = tuple(), 
        anti_filter: Tuple[str] = tuple()
    ):
    
    if filter is None:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv"))]
    else:
        csv_logs = [f for f in os.listdir(res_dir) if (
            os.path.isfile(os.path.join(res_dir, f)) and 
            f.endswith(".csv") and 
            all(sub in f for sub in filter)) and
            all(sub not in f for sub in anti_filter)
            ]

    csv_logs.sort()
    target_log_name = [l for l in csv_logs if "no_mutation" in l][-1]
    csv_logs.pop(csv_logs.index(target_log_name))
    target_log_path = os.path.join(res_dir, target_log_name)
    target_log = pd.read_csv(target_log_path)

    results_df = pd.DataFrame()

    total_inconsistencies = 0
    total_questions = 0
    total_success = 0
    total_answered = 0

    category_dict = {}
    mutation_dict = {}

    for log_name in csv_logs:
        mut_type = log_name.split('shot_')[1].split('.csv')[0]
                        
        log2_file_path = os.path.join(res_dir, log_name)
        log2 = pd.read_csv(log2_file_path) 

        target_log, log2 = DataLogHelper.standardize_two_df(target_log, log2)

        inconsistency_dict = DataLogHelper.compare_code_generation_dataframe_results(log1=target_log, log2=log2, benchmark = benchmark, task = task)

        # Adding results into the dataframe
        cleaned_mutation_name = DataLogHelper.clean_up_csv_name(log_name.replace('.csv', ''))

        # Metrics for model inconsistency calculation
        mutation_inconsistencies = inconsistency_dict['total_inconsistencies']
        mutation_questions = inconsistency_dict['total_inconsistency_comparisons']

        # Metrics for model accuracy calculation
        mutation_successes = inconsistency_dict['log2_success']
        mutation_answered = inconsistency_dict['log2_total_answered']

        results_df.loc[cleaned_mutation_name, "Inconsistency Score"] = f"{mutation_inconsistencies}/{mutation_questions} ({round(mutation_inconsistencies*100/mutation_questions, 2)}%)"
        results_df.loc['No Mutation', "Inconsistency Score"] = "N/A"
        results_df.loc['No Mutation', "Model Accuracy"] = f"{(inconsistency_dict['log1_success'])}/{inconsistency_dict['log1_total_answered']} ({round((inconsistency_dict['log1_success'])*100/inconsistency_dict['log1_total_answered'], 2)}%)"
        results_df.loc[cleaned_mutation_name, "Model Accuracy"] = f"{mutation_successes}/{mutation_answered} ({round(mutation_successes*100/mutation_answered, 2)}%)"

        if total_success == 0:
            total_success += inconsistency_dict['log1_success']
        
        if total_answered == 0:
            total_answered += inconsistency_dict['log1_total_answered']

        if 'model_ensemble' in log_name.lower() or "ensemble" not in log_name.lower() :
            total_inconsistencies += mutation_inconsistencies
            total_questions += mutation_questions
            total_success += mutation_successes
            total_answered += mutation_answered

        d: Dict = category_dict.get(mut_type, {})
        d['total_inconsistencies'] = d.get('total_inconsistencies', 0) + mutation_inconsistencies
        d['total_questions'] = d.get('total_questions', 0) + mutation_questions
        d['total_success'] = d.get('total_success', 0) + mutation_successes
        d['total_answered'] = d.get('total_answered', 0) + mutation_answered
        category_dict[mut_type] = d

        if not category_dict.get('no_mutation', None):
            d2 = {}
            d2['total_inconsistencies'] = 0
            d2['total_questions'] = 1
            d2['total_success'] = inconsistency_dict['log1_success']
            d2['total_answered'] = inconsistency_dict['log1_total_answered']
            category_dict['no_mutation'] = d2

        # adding results in mutation_dict, with the mutation name as key
        mutation_dict[cleaned_mutation_name] = {
            'total_inconsistencies': mutation_inconsistencies,
            'total_questions': mutation_questions,
            'total_success': mutation_successes,
            'total_answered': mutation_answered
        }
    
    results_df = pd.concat([
        results_df[results_df.index.str.lower().str.contains("no mutation")],

        results_df[
            ~results_df.index.str.lower().str.contains("ensemble") &
            ~results_df.index.str.lower().str.contains("no mutation")
        ],

        results_df[results_df.index.str.lower().str.contains("ensemble")]
    ])

    ## Adding aggregated second order results and atomic results
    for key, mut_dict in category_dict.items():
        mut_inconsistencies = mut_dict['total_inconsistencies']
        mut_questions = mut_dict['total_questions']
        mut_success = mut_dict['total_success']
        mut_answered = mut_dict['total_answered']
        results_df.loc[f"{key} Results", "Inconsistency Score"] = f"{mut_inconsistencies}/{mut_questions} ({round(mut_inconsistencies*100/mut_questions, 2)})"
        results_df.loc[f"{key} Results", "Model Accuracy"] = f"{mut_success}/{mut_answered} ({round(mut_success*100/mut_answered, 2)}%)"

        mutation_dict[f"{key} Results"] = mut_dict


    results_df.loc["Aggregated Results", "Inconsistency Score"] = f"{total_inconsistencies}/{total_questions} ({round(total_inconsistencies*100/total_questions, 2)})"
    results_df.loc["Aggregated Results", "Model Accuracy"] = f"{total_success}/{total_answered} ({round(total_success*100/total_answered, 2)}%)"


    return [
        results_df, 
        category_dict, 
        ]

In [6]:
model_dict = {
    "Qwen2.5-Coder-14B-Instruct" : "Qwen2.5-Coder-14B-Instruct",
    "gemma-3-12b-it": "Gemma-3-12b-it",
    "deepseek-reasoner": "DeepSeek-V3.2-Exp (Non-thinking Mode)",
    "LLama-3.1-8B": "LLama-3.1-8B",
    "gpt-5" : "GPT-5",
    "gpt-4o": "GPT-4o",
    "codestral-latest": "codestral-2508",
}

def obtain_benchmark_task_csv(benchmark: str, task: str) -> pd.DataFrame:

    final_df = pd.DataFrame()  # start with an empty DataFrame
    final_dict = {}

    # Iterating through each model in model_dict
    for k, m in model_dict.items():
        # print(k)
        res_dir = os.path.join(parent_dir, f"MuCoCo_experiment_results/{task}/{k}")
        try:
            res, category_dict = compare_logs_against_no_mutation(
                res_dir=res_dir, 
                benchmark = benchmark, 
                task = task, 
                filter=(benchmark, )
            )

        except FileNotFoundError:
            print(f"{res_dir} does not exist.")
            continue

        res_df = pd.DataFrame(res)

        res_df = res_df.add_prefix(f"{m} ")

        if final_df.empty:
            final_df = res_df
        else:
            final_df = pd.concat([final_df, res_df], axis=1)

        final_dict[m] = category_dict

    # final_df.to_csv("combined_results.csv", index=True, header=True)
    return final_df, final_dict


In [7]:
from tqdm import tqdm
import copy


tasks = {
    'mcq_inconsistency': ['CodeMMLU'],
    'input_prediction': ['HumanEval', "CruxEval"],
    'output_prediction': ['HumanEval', "CruxEval"],
    # 'code_generation': ['BigCodeBench', "HumanEval"],
    'code_generation': ["HumanEval"],

}

task_dict = {}
overall_dict = {}
all_benchmark_dict = {}
dfs = []

def combine_two_dictionaries(d1: dict, d2: dict) -> dict:
    # make it PURE (return a new merged dict)
    out = copy.deepcopy(d1)
    for k, inner2 in d2.items():
        if k not in out:
            out[k] = copy.deepcopy(inner2)              # shallow is enough at this level
        else:
            for kk, vv in inner2.items():
                out[k][kk] = out[k].get(kk, 0) + vv
    return out


for task, benchmarks in tqdm(tasks.items()):
    # Dictionary for storing results to aggregate by task
    task_d = {}

    print(f"Aggregating for {task} logs")
    for benchmark in benchmarks:

        print(f"Working on {benchmark} now...")
        final_df, aggregated_dict = obtain_benchmark_task_csv(benchmark, task)
        
        benchmark_dict = {}

        for model, mut_cat_dict in aggregated_dict.items():
            if "ensemble" in model:
                continue
            for mut_cat, res_dir in mut_cat_dict.items():
                # make a NEW dict here instead of aliasing res_dir
                if not benchmark_dict.get(mut_cat, None):
                    benchmark_dict[mut_cat] = res_dir.copy()
                else:
                    for key, val in res_dir.items():
                        benchmark_dict[mut_cat][key] += val
                    
        # building benchmark dict for aggregating results by benchmark
        d = all_benchmark_dict.get(benchmark, {})
        if not d:
            all_benchmark_dict[benchmark] = benchmark_dict
        else:
            new_d = combine_two_dictionaries(d, benchmark_dict) 
            all_benchmark_dict[benchmark] = new_d

for k, v in all_benchmark_dict.items():
    print(f"{k}: {v}")

  0%|          | 0/4 [00:00<?, ?it/s]

Aggregating for mcq_inconsistency logs
Working on CodeMMLU now...


 25%|██▌       | 1/4 [00:03<00:09,  3.06s/it]

Aggregating for input_prediction logs
Working on HumanEval now...
Working on CruxEval now...


 50%|█████     | 2/4 [00:47<00:54, 27.12s/it]

Aggregating for output_prediction logs
Working on HumanEval now...
Working on CruxEval now...


 75%|███████▌  | 3/4 [01:24<00:31, 31.95s/it]

Aggregating for code_generation logs
Working on HumanEval now...


100%|██████████| 4/4 [02:45<00:00, 41.47s/it]

-0.5
1.0
CodeMMLU: {'boolean_literal': {'total_inconsistencies': 78, 'total_questions': 210, 'total_success': 97, 'total_answered': 210}, 'no_mutation': {'total_inconsistencies': 0, 'total_questions': 7, 'total_success': 749, 'total_answered': 931}, 'commutative_reorder': {'total_inconsistencies': 48, 'total_questions': 248, 'total_success': 200, 'total_answered': 265}, 'constant_unfold': {'total_inconsistencies': 159, 'total_questions': 427, 'total_success': 188, 'total_answered': 427}, 'constant_unfold_add': {'total_inconsistencies': 201, 'total_questions': 427, 'total_success': 154, 'total_answered': 427}, 'constant_unfold_mult': {'total_inconsistencies': 186, 'total_questions': 427, 'total_success': 168, 'total_answered': 427}, 'demorgan': {'total_inconsistencies': 71, 'total_questions': 191, 'total_success': 98, 'total_answered': 191}, 'for2enumerate': {'total_inconsistencies': 52, 'total_questions': 405, 'total_success': 287, 'total_answered': 405}, 'for2while': {'total_inconsist

## Aggregating model results on BigCodeBench

In [12]:
import json
from collections import defaultdict

bigcodebench_res = {}
bigcodebench_res_dir = curr_dir / "bigcodebench_json"

mutations = ['Random', 'Sequential']
fields = ['total_inconsistencies', 'total_questions', 'total_success', 'total_answered']

# Use defaultdict to automatically initialize nested structure
bigcodebench_mut_dict = {m: defaultdict(int) for m in mutations}

for json_file in os.listdir(bigcodebench_res_dir):
    res_dir = bigcodebench_res_dir / json_file
    with open(res_dir, 'r') as file:
        data = json.load(file)
    
    model_name = model_dict[json_file.split('_')[0]]
    bigcodebench_res[model_name] = data
    
    # Aggregate directly while reading
    for m in mutations:
        if m in data:
            for f in fields:
                bigcodebench_mut_dict[m][f] += data[m].get(f, 0)

# Convert defaultdict back to regular dict if needed
bigcodebench_mut_dict = {m: dict(bigcodebench_mut_dict[m]) for m in mutations}

print(bigcodebench_mut_dict)

{'Random': {'total_inconsistencies': 2083, 'total_questions': 7744, 'total_success': 4437, 'total_answered': 7649}, 'Sequential': {'total_inconsistencies': 2139, 'total_questions': 7743, 'total_success': 4262, 'total_answered': 7627}}


In [22]:
import json
from collections import defaultdict

bigcodebench_res = {}
bigcodebench_res_dir = curr_dir / "bigcodebench_json"

mutations = ['Random', 'Sequential']
fields = ['total_inconsistencies', 'total_questions', 'total_success', 'total_answered']

# Use defaultdict to automatically initialize nested structure
bigcodebench_mut_dict = {m: defaultdict(int) for m in mutations}

for json_file in os.listdir(bigcodebench_res_dir):
    res_dir = bigcodebench_res_dir / json_file
    with open(res_dir, 'r') as file:
        data = json.load(file)
    
    model_name = model_dict[json_file.split('_')[0]]
    bigcodebench_res[model_name] = data
    
    # Aggregate directly while reading
    for m in mutations:
        if m in data:
            for f in fields:
                bigcodebench_mut_dict[m][f] += data[m].get(f, 0)

# Convert defaultdict back to regular dict if needed
bigcodebench_mut_dict = {m.lower(): dict(bigcodebench_mut_dict[m]) for m in mutations}

all_benchmark_dict['BigCodeBench'] = bigcodebench_mut_dict

for k, v in all_benchmark_dict.items():
    print(f"{k}: {v}")

CodeMMLU: {'boolean_literal': {'total_inconsistencies': 78, 'total_questions': 210, 'total_success': 97, 'total_answered': 210}, 'no_mutation': {'total_inconsistencies': 0, 'total_questions': 7, 'total_success': 749, 'total_answered': 931}, 'commutative_reorder': {'total_inconsistencies': 48, 'total_questions': 248, 'total_success': 200, 'total_answered': 265}, 'constant_unfold': {'total_inconsistencies': 159, 'total_questions': 427, 'total_success': 188, 'total_answered': 427}, 'constant_unfold_add': {'total_inconsistencies': 201, 'total_questions': 427, 'total_success': 154, 'total_answered': 427}, 'constant_unfold_mult': {'total_inconsistencies': 186, 'total_questions': 427, 'total_success': 168, 'total_answered': 427}, 'demorgan': {'total_inconsistencies': 71, 'total_questions': 191, 'total_success': 98, 'total_answered': 191}, 'for2enumerate': {'total_inconsistencies': 52, 'total_questions': 405, 'total_success': 287, 'total_answered': 405}, 'for2while': {'total_inconsistencies': 

# Consistency Error Rate and Accuracy by Mutation Operator

In [ ]:
import copy

benchmark_df = pd.DataFrame()
all_cat_dict = {}


for benchmark, mut_cat_dict in all_benchmark_dict.items():
    for mut_cat, res_dir in mut_cat_dict.items():
        d = all_cat_dict.get(mut_cat, {})
        if not d:
            all_cat_dict[mut_cat] = copy.deepcopy(res_dir)
        else:
            for key, value in d.items():
                d[key] += res_dir[key]
            all_cat_dict[mut_cat] = d

aggregated_mutation_inc = []
aggregated_mutation_acc = []

mutation_name_map = {
    "no_mutation": "no mutation",
    "boolean_literal": "boolean literal",
    "commutative_reorder": "commutative reorder",
    "constant_unfold": "constant unfold",
    "constant_unfold_add": "constant unfold add",
    "constant_unfold_mult": "constant unfold mult",
    "demorgan": "demorgan",
    "for2enumerate": "for-to-enumerate",
    "for2while": "for-to-while",
    "literal_format": "literal format",
    "random": "random",
    "sequential": "sequential"
}


total_inconsistencies = 0
total_questions = 0
total_successes = 0
total_answered = 0

for mut_cat, clean_name in mutation_name_map.items():
    res_dict = all_cat_dict[mut_cat]
    mut_inconsistencies = res_dict['total_inconsistencies']
    mut_questions = res_dict['total_questions']
    mut_success = res_dict['total_success']
    mut_answered = res_dict['total_answered']

    benchmark_df.loc[mut_cat, "Aggregated Inc."] =  f"{mut_inconsistencies}/{mut_questions} ({round(mut_inconsistencies*100/mut_questions, 2)})"
    benchmark_df.loc[mut_cat, "Aggregated Acc."] =  f"{mut_success}/{mut_answered} ({round(mut_success*100/mut_answered, 2)}%)"

    aggregated_mutation_inc.append(round(mut_inconsistencies*100/mut_questions, 2))
    aggregated_mutation_acc.append(round(mut_success*100/mut_answered, 2))
    
    log_cat = DataLogHelper.obtain_category(mut_cat)

    if mut_cat != "no_mutation":
        total_inconsistencies += mut_inconsistencies
        total_questions += mut_questions
        total_successes += mut_success
        total_answered += mut_answered

benchmark_df.loc["total", "Aggregated Inc."] = f"{total_inconsistencies}/{total_questions} ({round(total_inconsistencies*100/total_questions,2)})"
benchmark_df.loc["total", "Aggregated Acc."] = f"{total_successes}/{total_answered} ({round(total_successes*100/total_answered,2)})"
print(benchmark_df.to_string())

                           Aggregated Inc.        Aggregated Acc.
no_mutation                     0/42 (0.0)   19327/26988 (71.61%)
boolean_literal            380/4620 (8.23)      3659/4620 (79.2%)
commutative_reorder       835/6058 (13.78)     4030/6072 (66.37%)
constant_unfold          1396/9961 (14.01)     6764/9955 (67.95%)
constant_unfold_add      1696/9961 (17.03)     6624/9953 (66.55%)
constant_unfold_mult     1435/9961 (14.41)     6785/9957 (68.14%)
demorgan                  517/4447 (11.63)     3193/4443 (71.87%)
for2enumerate           1201/11591 (10.36)    8350/11584 (72.08%)
for2while               1490/11629 (12.81)    8262/11625 (71.07%)
literal_format            806/10236 (7.87)    7128/10228 (69.69%)
random                  6194/34736 (17.83)   23738/34634 (68.54%)
sequential              6005/34601 (17.35)   23430/34476 (67.96%)
total                 21955/147801 (14.85)  101963/147547 (69.11)


: 

In [24]:
import matplotlib
matplotlib.use('TkAgg')  # or 'Qt5Agg' depending on what's installed

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Data
data = {
    "mutation_type": list(mutation_name_map.values()),
    "Aggregated Mutation Inc.": aggregated_mutation_inc,
    "Aggregated Mutation Acc.": aggregated_mutation_acc,
}

df = pd.DataFrame(data)

# Setup
x = np.arange(len(df["mutation_type"]))  # positions
width = 0.38  # width of bars

fig, ax = plt.subplots(figsize=(12, 6))

# Bars
bars1 = ax.bar(x - width/2, df["Aggregated Mutation Inc."], width, label="Inconsistency (%)", color="tomato", alpha=0.8)
bars2 = ax.bar(x + width/2, df["Aggregated Mutation Acc."], width, label="Accuracy (%)", color="steelblue", alpha=0.85)

# Labels
ax.set_ylabel("Percentage (%)", fontsize=12)
ax.set_xlabel("Mutation Type", fontsize=12)
ax.set_title("Inconsistency vs Accuracy per Mutation Type", fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df["mutation_type"], rotation=45, ha="right", fontsize=9)
ax.set_ylim(0, 100)
ax.legend()

# Optional grid
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()

plt.savefig("plot.png")